In [42]:
import sys
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import TensorBoard
from tensorflow.keras.saving import save_model
from game_board import GameBoard

In [43]:
version = tf.__version__
print("TensorFlow version:", version)
python_version = sys.version
print("Python version:", python_version)

TensorFlow version: 2.20.0
Python version: 3.11.6 | packaged by conda-forge | (main, Oct  3 2023, 10:40:35) [GCC 12.3.0]


In [44]:
def create_model(input_shape, num_actions):
    model = Sequential()
    model.add(Dense(16, activation='relu', input_shape=input_shape))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(num_actions, activation='linear'))
    return model

In [45]:
model = create_model((16,), 14)
model.compile(optimizer='adam',
              loss='mse',
              metrics=['accuracy'])
model.summary()

/opt/conda/lib/python3.11/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_21 (Dense)                │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 14)             │           462 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,278 (4.99 KB)

 Trainable params: 1,278 (4.99 KB)

 Non-trainable params: 0 (0.00 B)

In [46]:
board = GameBoard()

gamma = 0.99

for t in range(100):
    board.initialize()
    print(f"Game {t+1} started")
    while True:
        state = board.get_status()
        player = state[14]
        old_total = state[6] if player == 0 else state[13]
        state_input = np.array(state).reshape(1, 16)
        q_values = model.predict(state_input, verbose=0)[0]

        legal_actions = board.get_playable_pits()
        best_action_index = np.argmax(q_values[legal_actions])
        action = legal_actions[best_action_index]

        board.play(action, player)

        reward = board.get_score()

        next_state = board.get_status()
        next_state_input = np.array(next_state).reshape(1, 16)
        next_q_values = model.predict(next_state_input, verbose=0)[0]
        legal_next_actions = board.get_playable_pits()
        if legal_next_actions:
            next_q = np.max(next_q_values[legal_next_actions])
        else:
            next_q = 0 

        target_q = q_values.copy()
        target_q[action] = reward + gamma * next_q
        model.fit(state_input, target_q.reshape(1, -1), verbose=0)
        print(state)
        if board.game_finish:
            print("Game finished")
            board.reset()
            break

Game 1 started
[4, 4, 4, 4, 4, 4, 0, 4, 4, 4, 4, 4, 4, 0, 1, 0]
[4, 0, 5, 5, 5, 5, 0, 4, 4, 4, 4, 4, 4, 0, 2, 0]
[4, 0, 5, 5, 5, 5, 0, 0, 5, 5, 5, 5, 4, 0, 1, 0]
[4, 0, 5, 0, 6, 6, 7, 1, 0, 5, 5, 5, 4, 0, 2, 0]
[4, 0, 5, 0, 0, 6, 7, 0, 0, 5, 5, 5, 4, 7, 1, 0]
[4, 0, 5, 0, 0, 0, 14, 1, 1, 6, 6, 0, 4, 7, 2, 0]
[4, 0, 5, 0, 0, 0, 14, 1, 0, 7, 6, 0, 4, 7, 1, 0]
[4, 0, 0, 1, 1, 1, 17, 0, 0, 7, 6, 0, 4, 7, 2, 0]
[5, 1, 1, 1, 1, 1, 17, 0, 0, 0, 7, 1, 5, 8, 1, 0]
[5, 1, 1, 0, 2, 1, 17, 0, 0, 0, 7, 1, 5, 8, 2, 0]
[6, 2, 2, 1, 2, 1, 17, 0, 0, 0, 7, 1, 0, 9, 1, 0]
[6, 2, 2, 0, 3, 1, 17, 0, 0, 0, 7, 1, 0, 9, 2, 0]
[0, 2, 2, 0, 3, 1, 17, 0, 0, 0, 7, 0, 0, 16, 1, 0]
[0, 2, 2, 0, 3, 0, 18, 0, 0, 0, 7, 0, 0, 16, 1, 0]
[0, 0, 3, 0, 3, 0, 19, 0, 0, 0, 7, 0, 0, 16, 2, 0]
[1, 1, 4, 1, 3, 0, 19, 0, 0, 0, 0, 1, 1, 17, 1, 0]
[1, 0, 5, 1, 3, 0, 19, 0, 0, 0, 0, 1, 1, 17, 2, 0]
Game saved: game_history.txt
[1, 0, 5, 1, 3, 0, 19, 0, 0, 0, 0, 1, 0, 18, 2, 0]
Game finished
Game 2 started
[4, 4, 4, 4, 4, 4, 0, 4, 4

In [47]:
save_model(model, 'mancala_model.keras')